# FP_NB_Silver_Transform
**Northbrook Lending (fictional UK lender) – Bronze ➜ Silver**

| | |
|---|---|
| **Source** | `FP_Bronze.dbo.*` – OneLake shortcuts to the mirrored tables of the `FP_Northbrook_OLTP` SQL database (no data copied into Bronze) |
| **Target** | `FP_Silver.dbo.*` – cleaned, conformed Delta tables |
| **Load pattern** | Full refresh for reference and transactional tables; **SCD Type 2** merge for `Customer` |

**What this notebook does**
1. **Clean**: trims text and turns blank strings into NULL; upper-cases postcodes, lower-cases e-mails.
2. **Conform types**: `smallint` → `int`, all decimals → `decimal(18,2)`, so downstream joins and sums behave consistently.
3. **Deduplicate**: keeps the latest row per business key (by `ModifiedAt`).
4. **Audit columns**: `_source_system`, `_run_id`, `_loaded_at`, `_row_hash` (SHA-256 of the business columns).
5. **SCD Type 2 on Customer**: changes to address, employment, income or credit score close the current row (`ValidTo`, `IsCurrent = false`) and add a new version. Contact-detail changes (e-mail, phone, name) are applied in place (Type 1).
6. **Schema drift**: new source columns flow through automatically (`overwriteSchema` / `mergeSchema`). See `FP_NB_SchemaDrift_Test`.

Lakehouse paths are resolved at run time, so the notebook works unchanged after deployment from Dev to Test.


In [ ]:
# Parameters – overridden by the FP_PL_Daily_Load pipeline
# Default value - the pipeline replaces it with its own run ID at run time; it stays empty when you run the notebook by hand
run_id = ""   # pipeline run id; a GUID is generated when the notebook is run by hand

In [ ]:
# ---------- Setup: imports, lakehouse paths and helper functions ----------
# uuid makes run IDs for manual runs; json builds the exit value returned to the pipeline
import uuid, json
# pandas - used to build the run summary table at the end
import pandas as pd
# Spark SQL functions such as col, trim and sha2 - imported as F
from pyspark.sql import functions as F
# Window - lets us rank rows within groups, used for de-duplication
from pyspark.sql.window import Window
# Spark data types - used to decide how each column is cleaned
from pyspark.sql.types import StringType, ShortType, ByteType, DecimalType
# DeltaTable - open an existing Delta table to run MERGE updates against it
from delta.tables import DeltaTable

# Use the pipeline run ID if one was passed in, otherwise make a random one for manual runs
run_id = run_id or str(uuid.uuid4())
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")   # allow MERGE to add new source columns

# Find the OneLake root folder of a lakehouse by name
def lakehouse_root(name: str) -> str:
    """OneLake root of a lakehouse in the *current* workspace (deployment-safe: no hard-coded IDs)."""
    try:
        # Preferred - ask Fabric for this lakehouse by name
        lh = notebookutils.lakehouse.get(name)
        # The lookup can return a dict or an object - read its properties either way
        props = lh["properties"] if isinstance(lh, dict) else lh.properties
        # Take its OneLake Tables path and cut off /Tables to leave the lakehouse root
        return props["oneLakeTablesPath"].rstrip("/").rsplit("/Tables", 1)[0]
    except Exception:
        # Fallback if the lookup fails - build the OneLake path from the current workspace name
        ws = notebookutils.runtime.context["currentWorkspaceName"]
        return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{name}.Lakehouse"

# Build the full path to a Delta table - root/Tables/dbo/TableName
def table_path(root: str, table: str, schema: str = "dbo") -> str:
    return f"{root}/Tables/{schema}/{table}"

# Find the Bronze and Silver lakehouse roots once for the whole run
BRONZE, SILVER = lakehouse_root("FP_Bronze"), lakehouse_root("FP_Silver")

# Standard clean-up applied to every table
def conform(df):
    """Trim text (blank -> NULL), widen small integers and standardise decimals."""
    # Check every column and apply a fix based on its data type
    for f in df.schema.fields:
        # Refer to the column in backticks so names with spaces or odd characters still work
        c = F.col(f"`{f.name}`")
        # Text - trim spaces, and turn empty strings into NULL
        if isinstance(f.dataType, StringType):
            df = df.withColumn(f.name, F.when(F.trim(c) == "", None).otherwise(F.trim(c)))
        # Small whole numbers - widen smallint and tinyint to int
        elif isinstance(f.dataType, (ShortType, ByteType)):
            df = df.withColumn(f.name, c.cast("int"))
        # Decimals - standardise to 18 digits with 2 decimal places so sums and joins line up
        elif isinstance(f.dataType, DecimalType):
            df = df.withColumn(f.name, c.cast("decimal(18,2)"))
    return df

# Remove duplicates so each business key appears once
def dedupe(df, key: str):
    """Keep only the latest version of each business key."""
    # One group per business key, newest ModifiedAt first, rows with no ModifiedAt last
    w = Window.partitionBy(key).orderBy(F.col("ModifiedAt").desc_nulls_last())
    # Number the rows in each group, keep row 1 - the newest - and drop the helper column
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

# Build one fingerprint column from a list of columns - used to spot rows that have changed
def hash_cols(cols):
    # Cast each column to text, swap NULLs for ~, join them with || and take a SHA-256 hash
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(f"`{c}`").cast("string"), F.lit("~")) for c in cols]), 256)

def add_audit(df):
    """Lineage / audit columns added to every Silver row."""
    # Business columns = every column that does not start with an underscore
    business_cols = [c for c in df.columns if not c.startswith("_")]
    # Add four columns - source system label, pipeline run ID, load time, and a SHA-256 hash of the business columns
    return (df.withColumn("_source_system", F.lit("FP_Northbrook_OLTP"))
              .withColumn("_run_id", F.lit(run_id))
              .withColumn("_loaded_at", F.current_timestamp())
              .withColumn("_row_hash", hash_cols(business_cols)))

# table-specific standardisation rules
# Each table name maps to a small function that takes a DataFrame and returns it fixed; tables not listed get no extra rule
RULES = {
    # Customer - upper-case postcodes and lower-case e-mail addresses
    "Customer": lambda d: d.withColumn("Postcode", F.upper("Postcode")).withColumn("Email", F.lower("Email")),
    # Branch - upper-case postcodes
    "Branch":   lambda d: d.withColumn("Postcode", F.upper("Postcode")),
}

# Print the run ID and the resolved lakehouse paths so each run is easy to check
print("Run id :", run_id)
print("Bronze :", BRONZE)
print("Silver :", SILVER)

In [ ]:
# ---------- Reference and transactional tables: clean, conform, dedupe, audit, full refresh ----------
# Tables to load, each mapped to its business key column - used for de-duplication
TABLES = {"Branch": "BranchID", "Product": "ProductID", "LoanApplication": "ApplicationID",
          "Loan": "LoanID", "Repayment": "RepaymentID", "Arrears": "ArrearsID"}
# Empty list that collects row counts for each table as it loads
metrics = []

# Process each table in turn
for table, key in TABLES.items():
    # Read the Bronze table - a shortcut to the mirrored source table
    raw = spark.read.format("delta").load(table_path(BRONZE, table))
    # Clean the data, inside out - conform types, apply any table rule, keep the latest row per key, then add the audit columns
    clean = add_audit(dedupe(RULES.get(table, lambda d: d)(conform(raw)), key))
    # Write to Silver as a full refresh - the whole table is replaced on every run
    (clean.write.format("delta").mode("overwrite")
          .option("overwriteSchema", "true")          # additive schema drift flows through
          .save(table_path(SILVER, table)))
    # Count rows in Bronze and in the new Silver table, so the difference shows how many duplicates were removed
    n_in = raw.count()
    n_out = spark.read.format("delta").load(table_path(SILVER, table)).count()
    # Record this table's counts for the run summary and the notebook exit value
    metrics.append({"table": table, "load_type": "Full refresh", "bronze_rows": n_in,
                    "silver_rows": n_out, "duplicates_removed": n_in - n_out})
    # Print a one-line summary of the row counts for this table
    print(f"{table:<16} bronze={n_in:>7,}  silver={n_out:>7,}")

In [ ]:
# ---------- Customer: SCD Type 2 ----------
# Tracked columns - a change to any of these keeps the old row as history and adds a new version
SCD2_TRACKED = ["AddressLine1", "Town", "Region", "Postcode", "EmploymentStatus", "AnnualIncome", "CreditScore"]
# Far-future end date given to current rows, so they never look expired
HIGH_DATE = F.lit("9999-12-31 00:00:00").cast("timestamp")
# Full path to the Silver Customer table
cust_path = table_path(SILVER, "Customer")

# Read Customer from Bronze
bronze_cust = spark.read.format("delta").load(table_path(BRONZE, "Customer"))
# Clean the data - conform types, apply Customer rules, keep the latest row per CustomerID, add audit columns
src = add_audit(dedupe(RULES["Customer"](conform(bronze_cust)), "CustomerID"))
# Second hash over the tracked columns only - if this changes, the customer needs a new version
src = src.withColumn("_scd_hash", hash_cols(SCD2_TRACKED))

# First load - the Silver Customer table does not exist yet
if not DeltaTable.isDeltaTable(spark, cust_path):
    # first load: one version per customer, valid from the date they became a customer
    first = (src.withColumn("CustomerVersion", F.lit(1))
                .withColumn("ValidFrom", F.col("CustomerSince").cast("timestamp"))
                .withColumn("ValidTo", HIGH_DATE)
                .withColumn("IsCurrent", F.lit(True)))
    # Create the Silver Customer table
    first.write.format("delta").save(cust_path)
    # Counts for this run - every customer is new on the first load
    scd = {"new_customers": first.count(), "new_versions": 0, "type1_updates": 0}
# Later loads - compare the source with the current rows in Silver
else:
    # Open the existing Silver Customer table
    tgt = DeltaTable.forPath(spark, cust_path)
    # Current rows only, with their hashes renamed so they can sit beside the source hashes
    cur = (tgt.toDF().filter("IsCurrent")
              .select("CustomerID", "CustomerVersion",
                      F.col("_scd_hash").alias("_cur_scd"), F.col("_row_hash").alias("_cur_row")))
    # snapshot the comparison before the target is modified
    # Left join keeps every source customer; the current-row columns are NULL when there is no match
    j = src.join(cur, "CustomerID", "left").localCheckpoint()
    # New customer - no current row exists in Silver yet
    new_cust = j.filter("_cur_scd IS NULL")
    # Changed customer - a tracked column differs, so a new history version is needed
    changed  = j.filter("_cur_scd IS NOT NULL AND _scd_hash <> _cur_scd")
    # Type 1 change - tracked columns unchanged but another column differs, so update in place
    type1    = j.filter("_cur_scd IS NOT NULL AND _scd_hash = _cur_scd AND _row_hash <> _cur_row")

    # 1) close the current version of customers whose tracked attributes changed
    # Match each changed customer to their current row in Silver
    (tgt.alias("t").merge(changed.select("CustomerID", "ModifiedAt").alias("c"),
                          "t.CustomerID = c.CustomerID AND t.IsCurrent = true")
        # Mark the old version as no longer current and end-date it at the time of the change
        .whenMatchedUpdate(set={"IsCurrent": "false", "ValidTo": "c.ModifiedAt"}).execute())

    # 2) Type 1: overwrite non-tracked attributes (e-mail, phone, name...) on the current version
    # Type 1 columns - every source column except the tracked ones and the key
    t1_cols = [c for c in src.columns if c not in SCD2_TRACKED and c != "CustomerID"]
    # Match each Type 1 customer to their current row in Silver
    (tgt.alias("t").merge(type1.select(*src.columns).alias("s"),
                          "t.CustomerID = s.CustomerID AND t.IsCurrent = true")
        # Copy every Type 1 column from the source onto the current row
        .whenMatchedUpdate(set={c: f"s.`{c}`" for c in t1_cols}).execute())

    # 3) insert new versions for changed customers, and version 1 for brand-new customers
    # Changed customers get the next version number, valid from the time of the change
    inserts = (changed.withColumn("CustomerVersion", F.col("CustomerVersion") + 1)
                      .withColumn("ValidFrom", F.col("ModifiedAt"))
               # Brand-new customers start at version 1, valid from their CustomerSince date
               .unionByName(new_cust.withColumn("CustomerVersion", F.lit(1))
                                    .withColumn("ValidFrom", F.col("CustomerSince").cast("timestamp")))
               # New rows are open-ended and current; then drop the helper comparison columns
               .withColumn("ValidTo", HIGH_DATE)
               .withColumn("IsCurrent", F.lit(True))
               .drop("_cur_scd", "_cur_row"))
    # Append the new rows to the Customer table; mergeSchema lets new source columns through
    inserts.write.format("delta").mode("append").option("mergeSchema", "true").save(cust_path)
    # Counts for this run
    scd = {"new_customers": new_cust.count(), "new_versions": changed.count(), "type1_updates": type1.count()}

# Re-read the Customer table after all changes, for the final counts
cust = spark.read.format("delta").load(cust_path)
# Record Customer counts for the run summary, including how many rows are current
metrics.append({"table": "Customer", "load_type": "SCD Type 2", "bronze_rows": bronze_cust.count(),
                "silver_rows": cust.count(), "current_rows": cust.filter("IsCurrent").count(), **scd})
# Show the SCD counts for this run
print(scd)

In [ ]:
# ---------- Run summary ----------
# Turn the per-table counts into a table, blank out missing values, make every value text so mixed types display, and show it
display(spark.createDataFrame(pd.DataFrame(metrics).fillna("").astype(str)))

In [ ]:
# End the notebook and hand a JSON summary back to the calling pipeline - run ID, status and per-table counts; default=str turns any non-JSON values into text
notebookutils.notebook.exit(json.dumps({"notebook": "FP_NB_Silver_Transform", "run_id": run_id,
                                        "status": "Succeeded", "tables": metrics}, default=str))